In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
from collections.abc import Iterator
from typing import cast

import numpy as np
import polars as pl
from IPython.display import display
from maite_datasets.multiobject_tracking import (
    MultiobjectTrackingTargetTuple,
    SingleFrameObjectTrackingTargetTuple,
    VideoFrameTuple,
)

from dataeval import Metadata
from dataeval.core import track_stats
from dataeval.protocols import DatasetMetadata, DatumMetadata, MultiobjectTrackingDatum, VideoFrame
from dataeval.quality import Outliers
from dataeval.types import SourceIndex

pl.Config.set_tbl_width_chars(160)

polars.config.Config

In [3]:
HEIGHT, WIDTH, N_FRAMES, N_SEQUENCES, N_TRACKS = 72, 96, 40, 12, 4
rng = np.random.default_rng(0)


def path(n_frames: int, noise: float) -> np.ndarray:
    """Box centers for one object drifting at constant velocity, observed with `noise` pixels of error."""
    start = rng.uniform([10, 10], [WIDTH - 30, HEIGHT - 30])
    velocity = rng.uniform(-0.6, 0.6, 2)
    return start + velocity * np.arange(n_frames)[:, None] + rng.normal(0, noise, (n_frames, 2))


def tracks_for(defect: str | None) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    """Each track as (frames it appears in, box center in each of them)."""
    tracks = {track: (np.arange(N_FRAMES), path(N_FRAMES, noise=0.6)) for track in range(N_TRACKS)}
    if defect == "flicker":
        frames = np.arange(0, N_FRAMES, 2)
        tracks[0] = (frames, tracks[0][1][frames])
    elif defect == "reused_id":
        frames = np.r_[0:3, N_FRAMES - 3 : N_FRAMES]
        tracks[0] = (frames, np.r_[path(3, 0.6), path(3, 0.6) + 40])
    elif defect == "jitter":
        tracks[0] = (np.arange(N_FRAMES), path(N_FRAMES, noise=4.0))
    return tracks


def frame_targets(defect: str | None) -> list[SingleFrameObjectTrackingTargetTuple]:
    """One target per frame, holding a 12-pixel box for every track present in it."""
    tracks = tracks_for(defect)
    targets = []
    for frame in range(N_FRAMES):
        boxes, ids = [], []
        for track, (frames, centers) in tracks.items():
            at = np.flatnonzero(frames == frame)
            if at.size:
                cx, cy = centers[at[0]]
                boxes.append([cx - 6, cy - 6, cx + 6, cy + 6])
                ids.append(track)
        targets.append(
            SingleFrameObjectTrackingTargetTuple(
                boxes=np.array(boxes, dtype=np.float32).reshape(-1, 4),
                labels=np.zeros(len(ids), dtype=np.int64),
                scores=np.ones(len(ids), dtype=np.float32),
                track_ids=np.array(ids, dtype=np.int64),
            )
        )
    return targets


class BlankVideo:
    """A stand-in for a decoded video: the right number of frames, all zeros."""

    def __iter__(self) -> Iterator[VideoFrame]:
        blank = np.zeros((3, HEIGHT, WIDTH), dtype=np.uint8)
        for index in range(N_FRAMES):
            yield VideoFrameTuple(pixels=blank, time_s=index / 30, pts=index, frame_index=index)


class TrackingDataset:
    """A minimal MAITE multi-object tracking dataset."""

    def __init__(self, defects: dict[int, str]) -> None:
        self.metadata = DatasetMetadata({"id": "planted-defects", "index2label": {0: "vehicle"}})
        self._data: list[MultiobjectTrackingDatum] = [
            (
                BlankVideo(),
                MultiobjectTrackingTargetTuple(frame_tracks=frame_targets(defects.get(index))),
                cast(DatumMetadata, {"id": f"sequence_{index}", "height": HEIGHT, "width": WIDTH}),
            )
            for index in range(N_SEQUENCES)
        ]

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, index: int) -> MultiobjectTrackingDatum:
        return self._data[index]


dataset = TrackingDataset({2: "flicker", 5: "reused_id", 7: "jitter"})

In [4]:
metadata = Metadata(dataset)
metadata.add_factors(track_stats(dataset), level="track", key="track_id")

exposure_ms = rng.normal(8.0, 0.4, metadata.level_counts["unit"])
exposure_ms[2 * N_FRAMES + 20] = 30.0
metadata.add_factors({"exposure_ms": exposure_ms}, level="unit")

print(metadata.level_counts)

{'sequence': 12, 'unit': 480, 'track': 48, 'instance': 1866}


In [5]:
try:
    Outliers().evaluate(metadata, factors=["labels"])
except ValueError as error:
    print(error)

Outliers.evaluate: {'labels': 'categorical'} cannot be thresholded: categorical factors have no location or scale, and ineligible columns are not quantities.


In [6]:
result = Outliers().evaluate(metadata)
display(result.aggregate_by_metric())

metric_name,Total
cat,u32
"""gap_fraction""",2
"""n_appearances""",2
"""total_gap_length""",2
"""track_length""",2
"""duration_s""",1
…,…
"""frame_span""",1
"""mean_speed""",1
"""n_gaps""",1


In [7]:
signals = ["n_gaps", "gap_fraction", "speed_variance", "exposure_ms"]
focused = Outliers().evaluate(metadata, factors=signals)
display(focused.data().select("item_index", "target_index", "level", "metric_name", "metric_value", "percentile"))

item_index,target_index,level,metric_name,metric_value,percentile
i64,i64,cat,cat,f64,f64
2,0,"""track""","""gap_fraction""",0.487179,96.875
2,0,"""track""","""n_gaps""",19.0,98.958333
2,20,"""unit""","""exposure_ms""",30.0,99.895833
5,0,"""track""","""gap_fraction""",0.85,98.958333
7,0,"""track""","""speed_variance""",11.993047,98.958333


In [8]:
print(focused.outliers)

flickering = metadata.rows_at("track").filter((pl.col("item_index") == 2) & (pl.col("track_id") == 0))
display(flickering.select("item_index", "track_id", "n_appearances", "track_duration", "n_gaps", "gap_fraction"))

{SourceIndex(2, 0, 'track'): ['gap_fraction', 'n_gaps'], SourceIndex(2, 20, 'unit'): ['exposure_ms'], SourceIndex(5, 0, 'track'): ['gap_fraction'], SourceIndex(7, 0, 'track'): ['speed_variance']}


/tmp/ipykernel_11199/1967393908.py:3: UserWarning: `exposure_ms`, `jitter_rate`, `mean_speed`, `net_displacement`, `pts`, `speed_variance`, `straightness_index` and `time_s` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"exposure_ms": [...]} to control this.
  flickering = metadata.rows_at("track").filter((pl.col("item_index") == 2) & (pl.col("track_id") == 0))


item_index,track_id,n_appearances,track_duration,n_gaps,gap_fraction
i64,i64,i64,i64,i64,f64
2,0,20,39,19,0.487179


In [9]:
# TEST ASSERTION CELL ###
assert focused.outliers == {
    SourceIndex(2, 0, "track"): ["gap_fraction", "n_gaps"],
    SourceIndex(5, 0, "track"): ["gap_fraction"],
    SourceIndex(7, 0, "track"): ["speed_variance"],
    SourceIndex(2, 20, "unit"): ["exposure_ms"],
}
assert "labels" not in set(result.data()["metric_name"].cast(pl.Utf8))